# Notebook 1 - Sinh dữ liệu thực nghiệm

Sinh 100 subtest (5 kích thước x 5 density x 4 seed), từ 10 đến 1000 elements. Mỗi subtest được lưu thành một file JSON riêng trong thư mục `data/` của repository.

Số subset được giữ ở mức khoảng 10% số elements để dữ liệu lớn nhưng vẫn thực tế cho bước đánh giá có timeout bảo vệ.


In [16]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from data_generator import generate_instance, instance_to_dict

In [17]:
DATA_DIR = REPO_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

NUM_ELEMENTS = [10, 20, 40, 80, 100]
DENSITIES = [0.1, 0.3, 0.5, 0.7, 0.9]
REPETITIONS = 4
BASE_SEED = 2026

generated = []
for num_elements in NUM_ELEMENTS:
    num_subsets = max(8, round(num_elements * 0.1))
    for density in DENSITIES:
        density_tag = f"{density:.1f}".replace(".", "")
        for repetition in range(REPETITIONS):
            seed = BASE_SEED + num_elements * 1000 + int(density * 100) * 10 + repetition
            instance = generate_instance(
                num_elements=num_elements,
                num_subsets=num_subsets,
                density=density,
                seed=seed,
            )
            payload = instance_to_dict(instance)
            payload["metadata"] = {
                "num_elements": num_elements,
                "num_subsets": num_subsets,
                "density": density,
                "repetition": repetition,
                "seed": seed,
            }
            filename = f"n{num_elements:04d}_d{density_tag}_r{repetition:02d}.json"
            (DATA_DIR / filename).write_text(
                json.dumps(payload, indent=2) + "\n", encoding="utf-8"
            )
            generated.append(filename)

print(f"Generated {len(generated)} subtest files in {DATA_DIR}")


Generated 100 subtest files in /Users/vnpq/_MLDM/Advanced Algo & programming/MinimumCoverSet/data


In [18]:
from collections import Counter

counts = Counter(
    json.loads(path.read_text(encoding="utf-8"))["metadata"]["num_elements"]
    for path in DATA_DIR.glob("n*.json")
)
print(counts)
print(f"Tong so file JSON: {sum(counts.values())}")


Counter({10: 20, 80: 20, 40: 20, 20: 20, 100: 20})
Tong so file JSON: 100
